# MAE 6246 - Homework 7, Part 2: Furuta Pendulum Controller Design

**Name:** _Enter your name._  
**Due:** Friday, November 13, 2026, at 10:00 AM (Eastern Time)  
**Late penalty:** 1 point per 10 minutes late.  
**Part 2:** 60 points. **Full assignment:** 100 points, including handwritten Part 1 (40 points).

This notebook contains the complete Part 2 statement, design specifications, experiments, and response fields. `homework07.pdf` contains only the handwritten Part 1 assignment. Submit your clearly handwritten Part 1 PDF and this completed notebook through Blackboard.

Keep all requested plots, numerical tables, written responses, and the embedded video in the notebook. No separate MP4 is required. The course policy on generative AI applies; include the required supplementary record when AI materially influences your work.

Complete the point-mass Riccati derivation entirely in your handwritten Part 1 PDF.

Keep this notebook, `mjsim.py`, and `hw07_support.py` in the same folder. In Colab, the setup cell downloads the two Python files automatically if they are missing. Use the supplied Furuta-enabled `mjsim.py`; the beam-only Week 11 file is insufficient. The notebook contains provided design commands and evaluation tools. **Your work is to choose, test, justify, and refine the controllers**, and complete the response cells and marked choice cells.

## Setup and fixed experiment

Use the Homework 6 Furuta pendulum and coordinate conventions. The state and input are

$$X=[\phi,\theta,\dot\phi,\dot\theta]^T,\qquad u=\text{arm torque [N m]},$$

where $\theta=0$ is upright. The target is $X_\star=0$, $u_\star=0$. All four states are available for feedback; **no observer is required**. Angles and angular rates are in radians and rad/s internally.

`hw.load_plant()` loads the supplied Furuta-enabled wrapper with `equilibrium="upright"`, `dt=0.002`, `u_limit=0.20`, and `physics_dt=0.001`. The wrapper provides $A,B,C,D$ linearized about the upright equilibrium. Design a constant gain using this linear model, then apply

$$u_{\rm requested}=u_\star-K(X-X_\star),\qquad
u_{\rm applied}=\operatorname{clip}(u_{\rm requested},-0.20,0.20).$$

The simulator applies saturation and holds the applied torque for each 2 ms control sample. MuJoCo advances with a 1 ms physics step. Each experiment lasts the full 8 s. Keep the plant, timing, torque limit, grid, and pass criteria fixed. Do not wrap angles, alter physical parameters, reset states within a trajectory, or add swing-up, switching, or gain scheduling. Each design uses one fixed gain for every initial condition.

The setup cell installs missing dependencies. The supplied wrapper was checked with MuJoCo 3.17.0.

In [ ]:
import importlib.util
import subprocess
import sys
import os
from pathlib import Path

packages = {"mujoco": "mujoco==3.17.0", "numpy": "numpy", "scipy": "scipy",
            "matplotlib": "matplotlib", "imageio_ffmpeg": "imageio-ffmpeg"}
missing = [package for module, package in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
from urllib.request import urlretrieve
helper_base = "https://raw.githubusercontent.com/fdcl-gwu/MAE6246/main/notebooks/homework07/"
for filename in ["mjsim.py", "hw07_support.py"]:
    if not Path(filename).is_file():
        urlretrieve(helper_base + filename, filename)
        print("Downloaded", filename, "from the course repository.")
if sys.platform.startswith("linux") and not os.environ.get("DISPLAY"):
    os.environ.setdefault("MUJOCO_GL", "egl")

import numpy as np
import matplotlib.pyplot as plt
import mujoco
from scipy.linalg import solve_continuous_are
from scipy.signal import place_poles
from IPython.display import display, Markdown, Video
import mjsim
import hw07_support as hw

np.set_printoptions(precision=6, suppress=True)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})
print("MuJoCo:", mujoco.__version__, "(validated with 3.17.0)")
assert "furuta" in mjsim.available_dynamics(), "Use the supplied Furuta-enabled mjsim.py."
dyn = hw.load_plant()
A, B, C, D = dyn.A, dyn.B, dyn.C, dyn.D



## Part 2(a) - Controllability and open-loop poles (6 points)

Construct the controllability matrix, report its rank, and compute the open-loop poles of the upright linear model.

In [ ]:
# YOUR CODE HERE: construct [B, AB, A^2 B, A^3 B].
Ctrb = None
if Ctrb is None:
    raise NotImplementedError("Construct the controllability matrix.")
print("Controllability rank:", np.linalg.matrix_rank(Ctrb))
print("Open-loop poles:", np.linalg.eigvals(A))


**Response 2(a):** _Report the controllability matrix, its rank, and the open-loop poles._

### Local-design requirements

Test both selected controllers in **MuJoCo** for 8 s from $X_0=[10^\circ,5^\circ,0,0]^T$, converted to radians. Each selected controller must meet these requirements:

| Quantity | Actual constraint |
|---|---|
| Applied torque throughout the experiment | $\lvert u\rvert\le0.20$ N m |
| Arm angle at every sample from 4 to 8 s | $\lvert\phi\rvert\le2^\circ$ |
| Pendulum angle at every sample from 4 to 8 s | $\lvert\theta\rvert\le1^\circ$ |

There are no additional local angular-rate or transient-excursion requirements. The supplied evaluator checks these three limits. The results table displays the limits beside the measured values.

For each selected controller, show the MuJoCo state and applied-torque histories and the results table. The controller factory and reporting functions below are supplied. As in the ball-and-beam example, `make_controller(dyn, K)` returns the callback `controller(t, xhat)`. With `observer=None`, the callback receives the actual state.

In [ ]:
def make_controller(dyn, K):
    x_eq, u_eq = dyn.x_eq, dyn.u_eq
    def controller(t, xhat):
        return u_eq - K @ (xhat - x_eq)
    return controller


def local_test(K):
    mj = mjsim.simulate(dyn, hw.NOMINAL_X0, hw.T_FINAL,
                        make_controller(dyn, K), observer=None)
    return {"mj": mj, "metrics": hw.assess(mj, local=True)}


def metrics_table(records):
    # Angle errors are maxima over every recorded sample from 4 to 8 s.
    rows = ["| Design | Arm error [deg] | Pendulum error [deg] | Peak torque [N m] | Pass |",
            "|---|---:|---:|---:|:---:|",
            "| **Required limits** | **<= 2.00** | **<= 1.00** | **<= 0.200** | |"]
    for name, rec in records.items():
        m = rec["metrics"]
        rows.append(f"| {name} | {m['tail_max_deg'][0]:.3f} | "
                    f"{m['tail_max_deg'][1]:.3f} | {m['peak_torque_Nm']:.4f} | "
                    f"{m['passed']} |")
    display(Markdown("\n".join(rows)))


## Part 2(b) - Pole placement (12 points)

1. Compute the baseline gain for poles $\{-1,-1.25,-1.5,-1.75\}$ in s$^{-1}$. Add **one tuned set** of four distinct stable poles. Complex poles must occur in conjugate pairs. Use `place_poles(A, B, poles).gain_matrix`.
2. Compare the two designs using the supplied results table. Select a tuned gain meeting the three local constraints and show its MuJoCo state and torque histories.

Report the gains and achieved closed-loop poles, and briefly explain your pole choice. The baseline is intentionally untuned and need not meet the local constraints. You may try further candidates if needed.

In [ ]:
# YOUR CHOICES HERE: retain baseline and add one tuned pole set.
pp_trials = {"baseline": hw.BASE_POLES.copy()}
selected_pp = None  # Replace with the name of your tuned design.

if len(pp_trials) < 2 or selected_pp not in pp_trials or selected_pp == "baseline":
    raise NotImplementedError("Add a tuned pole set and select it by name.")
assert np.allclose(np.sort_complex(pp_trials["baseline"]), np.sort_complex(hw.BASE_POLES))


In [ ]:
pp_gains, pp_local = {}, {}
for name, poles in pp_trials.items():
    K = place_poles(A, B, poles).gain_matrix
    pp_gains[name] = K
    pp_local[name] = local_test(K)
    print(name, "K:", K, "closed-loop poles:", np.linalg.eigvals(A - B @ K))
metrics_table(pp_local)
K_baseline = pp_gains["baseline"]
K_pp = pp_gains[selected_pp]
assert pp_local[selected_pp]["metrics"]["passed"], "Retune the selected PP design."
mjsim.plot_response(pp_local[selected_pp]["mj"], labels=["MuJoCo"], estimates=False,
                    title="Selected pole-placement design")
plt.show()


**Response 2(b):** _Briefly explain your tuned pole choice and whether it meets the three local constraints._

## Part 2(c) - LQR (14 points)

Choose diagonal $Q\succ0$ and a positive $1\times1$ weight $R=[r]$ for

$$J=\frac12\int_0^\infty\left(X^TQX+r u^2\right)\,dt.$$

1. Test **two weight choices** producing different gains, using `P = solve_continuous_are(A, B, Q, R)` and `K = np.linalg.solve(R, B.T @ P)`.
2. Compare their local results and select an LQR gain meeting the three local constraints. Show the selected $P$, $K$, closed-loop poles, and MuJoCo state and torque histories.

Briefly explain the relative state and input penalties in your selected weights. Multiplying all weights by one common factor does not count as a different controller. You may try further candidates if needed.

In [ ]:
# YOUR CHOICES HERE. Each entry is a dictionary with name, Q, and R.
# Format: {"name": "trial_1", "Q": np.diag([...]), "R": np.array([[...]])}
lqr_trials = []
selected_lqr = None  # Replace with the name of your selected trial.

if len(lqr_trials) < 2 or selected_lqr not in [d["name"] for d in lqr_trials]:
    raise NotImplementedError("Supply two LQR weight choices and select one by name.")


In [ ]:
lqr_gains, lqr_matrices, lqr_local = {}, {}, {}
for trial in lqr_trials:
    name, Q, R = trial["name"], np.asarray(trial["Q"]), np.asarray(trial["R"])
    assert Q.shape == (4, 4) and R.shape == (1, 1)
    assert np.allclose(Q, np.diag(np.diag(Q))) and np.all(np.diag(Q) > 0)
    assert R[0, 0] > 0
    P = solve_continuous_are(A, B, Q, R)
    K = np.linalg.solve(R, B.T @ P)
    lqr_gains[name], lqr_matrices[name] = K, (P, Q, R)
    lqr_local[name] = local_test(K)
    print(name, "Q diagonal:", np.diag(Q), "R:", R, "K:", K)
    print("Closed-loop poles:", np.linalg.eigvals(A - B @ K))
assert len(lqr_gains) == len(lqr_trials), "Use distinct trial names."
assert all(not np.allclose(k1, k2) for i, k1 in enumerate(lqr_gains.values())
           for k2 in list(lqr_gains.values())[i+1:]), "Use different gains."
metrics_table(lqr_local)
K_lqr = lqr_gains[selected_lqr]
P_lqr, Q_lqr, R_lqr = lqr_matrices[selected_lqr]
print("Selected P:\n", P_lqr)
assert lqr_local[selected_lqr]["metrics"]["passed"], "Retune the selected LQR design."
mjsim.plot_response(lqr_local[selected_lqr]["mj"], labels=["MuJoCo"], estimates=False,
                    title="Selected LQR design")
plt.show()


**Response 2(c):** _Briefly explain your selected state and input weights and whether the resulting controller meets the three local constraints._

## Part 2(d) - Expand a sampled recovery set (20 points)

Test the baseline, selected PP controller, and selected LQR controller on the supplied 63-point grid. Each case starts at $X_0=[0,\theta_0,0,\dot\theta_0]^T$:

- $\theta_0=-40,-30,\ldots,40$ degrees;
- $\dot\theta_0=-120,-80,\ldots,120$ degrees/s.

The supplied grid evaluator uses these fixed recovery criteria:

| Quantity | Actual constraint |
|---|---|
| Applied torque for the full 8 s | $\lvert u\rvert\le0.20$ N m |
| Excursions for the full 8 s | $\lvert\phi\rvert\le120^\circ$, $\lvert\theta\rvert\le60^\circ$ |
| Angles at every sample from 6 to 8 s | $\lvert\phi\rvert\le2^\circ$, $\lvert\theta\rvert\le1^\circ$ |
| Angular rates at every sample from 6 to 8 s | $\lvert\dot\phi\rvert,\lvert\dot\theta\rvert\le5^\circ$/s |

All recorded states and inputs must be finite. These recovery criteria are separate from the simpler local test in 2(b)-(c). Do not change the evaluator, initial-state grid, model, or simulation settings.

Use the supplied code to show the three recovery maps and their successful-point counts. Select one fixed controller that retains all baseline successes and adds at least **10** successful points. You may retune your earlier choices if needed; do not use a different gain for each grid point.

In [ ]:
grid_gains = {"Baseline": K_baseline, "PP": K_pp, "LQR": K_lqr}
grid_results = {}
for label, K in grid_gains.items():
    print("Testing", label, "on 63 initial states ...", flush=True)
    grid_results[label] = hw.evaluate_grid(dyn, make_controller(dyn, K))
    hw.plot_grid(grid_results[label], title=label)
    plt.show()

rows = ["| Design | Successful / 63 | Added vs. baseline | Lost vs. baseline | Target met |",
        "|---|---:|---:|---:|---|"]
for label, result in grid_results.items():
    change = hw.compare_recovery(grid_results["Baseline"], result)
    rows.append(f"| {label} | {result['count']} | {change['added']} | "
                f"{change['lost']} | {change['target_met']} |")
display(Markdown("\n".join(rows)))


### Final design target

Select either **PP** or **LQR**. Its recovery map must contain every successful baseline point and at least **10 additional successful points**. It must also pass the original local test. The target concerns inclusion of sampled successful points, not merely a larger total count that hides losses.

In [ ]:
# YOUR CHOICE HERE: "PP" or "LQR".
final_method = None
if final_method not in ("PP", "LQR"):
    raise NotImplementedError("Select PP or LQR as your final design.")


In [ ]:
K_final = grid_gains[final_method]
change = hw.compare_recovery(grid_results["Baseline"], grid_results[final_method])
print("Final design:", final_method, "K:", K_final, "recovery change:", change)
assert hw.assess(local_test(K_final)["mj"], local=True)["passed"]
assert change["target_met"], "Retune: retain all baseline successes and add at least 10."


**Response 2(d):** _State your final controller, the number of added points, and whether any baseline successes were lost._

## Part 2(e) - Demonstrate the improvement (8 points)

Choose one grid point where the final controller succeeds and the baseline fails. Use the supplied code to compare their MuJoCo histories and embed the final controller's 8-second animation.

In a short paragraph, identify why the baseline fails and explain why a finite grid on a two-dimensional initial-state slice is not a proof of the full region of attraction.

In [ ]:
# YOUR CHOICES HERE: one newly successful point, in degrees and degrees/s.
demo_theta_deg = None
demo_rate_deg_s = None
if demo_theta_deg is None or demo_rate_deg_s is None:
    raise NotImplementedError("Choose a newly recovered point from the fixed grid.")


In [ ]:
js = np.flatnonzero(np.isclose(hw.THETA_DEG, demo_theta_deg))
is_ = np.flatnonzero(np.isclose(hw.RATE_DEG_S, demo_rate_deg_s))
assert len(js) == len(is_) == 1, "Choose an exact fixed-grid point."
i, j = is_[0], js[0]
assert not grid_results["Baseline"]["mask"][i, j]
assert grid_results[final_method]["mask"][i, j]
x_demo = np.deg2rad([0., demo_theta_deg, 0., demo_rate_deg_s])
demo_baseline = mjsim.simulate(dyn, x_demo, hw.T_FINAL, make_controller(dyn, K_baseline))
demo_final = mjsim.simulate(dyn, x_demo, hw.T_FINAL, make_controller(dyn, K_final))
print("Baseline criteria:", hw.assess(demo_baseline))
print("Final criteria:", hw.assess(demo_final))
mjsim.plot_response([demo_baseline, demo_final], labels=["Baseline", "Final"],
                    estimates=False, title="A newly recovered initial condition")
plt.show()


In [ ]:
video_path = mjsim.animate(demo_final, "homework07_furuta_control.mp4", fps=30)
display(Video(video_path, embed=True))


**Response 2(e):** _Identify the failed baseline condition and explain the limitation of the sampled recovery map._

## Submission checklist

- [ ] My clearly handwritten Part 1 PDF contains all seven subproblems.
- [ ] I reported the controllability matrix, rank, and open-loop poles.
- [ ] I tested the baseline plus one tuned pole set and two different LQR gains.
- [ ] Both selected controllers meet the displayed local constraints; their tables and MuJoCo histories are included.
- [ ] I included three recovery maps and the final controller's added/lost-point counts.
- [ ] I demonstrated a newly recovered point and embedded the final-controller video.
- [ ] I completed the short response cells, restarted the kernel, ran all cells, saved, and reopened the notebook.

## References

- [SciPy pole placement](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.place_poles.html)
- [SciPy continuous-time algebraic Riccati solver](https://docs.scipy.org/doc/scipy/reference/generated/scipy.linalg.solve_continuous_are.html)
- [MuJoCo Python interface](https://mujoco.readthedocs.io/en/stable/python.html)